# Artificial Intelligence and Machine Learning for Cyber Security Module 1

## Introduction

Welcome to the course **Artificial Intelligence and Machine Learning for Cybersecurity (CECS 7727)**. This course focuses on the emerging topic of how to apply machine learning and artificial intelligence to cybersecurity. With a combination of hands-on excercises the student will develop a deep understanding of the tools available in artificial intelligence and machine learning and how can they be used to defend, protect and investigate the security of a computer system, a network or an application.

## Jupyter Notebook

Throughout this course we will be using sample **Jupyter Notebooks**. The datasets used will be related to cybersecurity. You do not need to install your own Jupyter Notebook server on your computer, but you can do it if you want to. These files should work in your own installation as long as you have all the libraries installed. The advantage of using google colaborate is that you can copy these files and work on them on your own google drive

## Read and press play

Next to each code cell there is a **play** button. You should play each cell in order. It is not necesary to read all the text cells above them, but they give you context. Not all cells will produce an output. It is important to play them in order. You could also hit the button that says **Run All** in the menu bar up top. Then you can check the outputs.

# Importing Libraries

The libraries or modules incorporate frameworks, functions, variables, and other tools that help us do the statistical analysis or artificial intelligence modeling we are trying to acomplish. In this example we are going to do a simple **linear regression** using the **Scikit** library.
- **Sckikit-learn** also provides support for other machine learning tools.
- We will manage the data as **dataframes** using the **Pandas** module.
- **Pandas** is the main library used for data manipulation and analysis of structured data.
- **Numpy** harnesses the power of multi-dimensional arrays, matrices and their mathematical operaitons.
- Finally, **Matplotlib** is a visualization library to create charts, plots, graphs and almost any graphical representation of data.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Downloading the dataset

In order for machine learning, and many artificial intelligence tools, to work we need a dataset for training or studying. There are many dataset repositories online such as **Zenodo**. In this case we foudn the dataset in **GitHub**. You could capture your own dataset using tools such as **Ghidra**, **Tshark** or your computer's **System Monitor**.

Check this **references** for ways to generate your own datasets.
[1] [https://www.cyberforensicacademy.com](https://www.cyberforensicacademy.com/blog/top-10-cyber-forensic-tools-used-by-experts)
[2] [https://www.reddit.com](https://www.reddit.com/r/hacking/comments/1gqkbdb/best_reverse_engineering_tools/)
[3] [https://medium.com](https://medium.com/hootsuite-engineering/anomaly-based-intrusion-detection-system-using-machine-learning-a18e88694ce0)
[4] [https://docs.suricata.io](https://docs.suricata.io/en/suricata-7.0.13/rules/datasets.html)
[5] [https://research.cec.sc.edu](https://research.cec.sc.edu/files/cyberinfra/files/Lab%2011%20-%20Preprocessing%20of%20Zeek%20Output%20Logs%20for%20Machine%20Learning.pdf)
[6] [https://www.varonis.com](https://www.varonis.com/blog/malware-analysis-tools)

The datasets are loaded on the current directory on our google drive.

In Jupyter Notebooks any command line that is preceded by an exclamation sign (`! ls`) is a command line prompt. These commands are working on the underlying computer not on the python interpreter.

In [ ]:
# Download dataset
! rm -rf *.csv*
! wget -q https://raw.githubusercontent.com/notsodubeyous/IoT-Network-Intrusion-Detection-System-UNSW-NB15/refs/heads/master/datasets/UNSW_NB15.csv

# Loading the dataset into notebook
**Pandas** (with the alias pd) allows for the manipulation of structured data, or multidimensional arrays. The function **pd.read_csv** is the pandas function for reading csv files.
The tag *header=0* tells the function that the first line contain the header names. It is important manually observe the data we are using before coding so that we can prevent future errors.
- Always take a look at the data!

Now the data is loaded into a dataframe called *df* with its own headers. The command `df.info()` provides a sample view of the columns, data types and number of data points.

In [ ]:
# Load dataset
data = pd.read_csv('UNSW_NB15.csv',header=0)
df = pd.DataFrame(data)
df.info()

# Data Cleaning

During data cleaning we eliminate potential errors, such as type errors, empty cells or invalid values (NaN, Inf, Blank). We should examine the data using commands such as **info()**, **.describe()**, **.sample(n)**, **.tail(n)**, **.head(n)** to make sure the data is fit for modeling.

In this case we use `df.describe()` to get a sense of the range and statistics of each column.

In [ ]:
# Clean data
numeric_cols = ['dur', 'sbytes', 'spkts']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=numeric_cols)
df.describe()

# Preparing the Features

Features are also called the attributes, the variables or the input we will be using in our model. In a **Linear Regression** we are modeling a line that best fit the data. A line is described by two parameters (slope and intercept). Therefore two features are needed to create a sensible model of a **Linear Regression**.

In this case we are flattening our data, that means removing any additional dimensionality or column dependence, and also filtering to restrict our data to non-infinite values. We are also removing data that is too few and too far away, thats where the filter `X.flatten() < 2e5` comes from. These would be considered **outliers**.

The **features** we chose are connection duration (dur) and source bytes transfered (sbytes). The question we are trying to answer is: *Can we estimate how long a network connection lastd based on the amount of data transfered from the source?*

We would expect these to variables to show a **correlation**, that is when we increase one, the other will increase proportionaly. Also there is an expected minimum duration regardless of data size due to the transfer protocol themselves, therefore we expect our model to have both a slope and an intercept.

In [ ]:
# Prepare features
X = df[['sbytes']].values
y = df['dur'].values
mask = np.isfinite(X.flatten()) & np.isfinite(y) & (X.flatten() < 2e5)
X, y = X[mask], y[mask]
print("Data Size:",len(X), len(y))

# Sampling the data

This is a huge data set as you saw in a previous cell, almost 200,000 data points. We will be using a sample size of 50,000 points or the size of our filtered data set, whichever is smaller. Using `np.randomchoice()` we restrict our data set randomly to avoid introducing a bias.

In `train_test_split()` our data set is divided into a dataset used to train the model, and another smaller set (`test_size=0.2`) which will be used to test our model. The value `random_state=42` restricts our seed to the random number generator to obtain the same value during training and testing in case we need to compare results to avoid variability.

In [ ]:
# Sample for speed
sample_size = min(50000, len(X))
indices = np.random.choice(len(X), sample_size, replace=False)
X, y = X[indices], y[indices]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Sample Size:",len(X_train),"Test Size:",len(X_test))

# Training the Model

We define our model `LinearRegresion()` and then do a fit using `mode.fit(X_train, y_train)`. We are training our model using the training dataset chosed randomly in the previous cell.

We also print the values of the correlation.

In [ ]:
# Train model
model = LinearRegression()
model.fit(X_train, y_train)
print(f"Slope (Coefficient): {model.coef_[0]:.6f}")
print(f"Intercept: {model.intercept_:.2f}")

# Evaluating the model

Using the test values we can determine how good our model is and use that as a predictor of success or failure in future cases. We want to know, how well does our data predict values it has not been trained on. There are many metrics for measuring errors.

In [ ]:
# Evaluate
y_pred = model.predict(X_test)
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.6f}")

# Visualization

Finally we present a graph with the model and the data. We can ask a few questions about wether the data is well represented by a line. *Do we need to filter out other components?* *Is the data that does not fit a different **population** or does our model needs more parameters?* *Is there another variable that we need to restrict?*
An even better question for machine learning is *How can we use the features in our data set to predict behavior and detect anomalies?*

In [ ]:
# Visualize
plt.figure(figsize=(10, 6))
sample_idx = np.random.choice(len(X_test), min(2000, len(X_test)), replace=False)
plt.scatter(X_test[sample_idx], y_test[sample_idx], color='blue', alpha=0.3, s=10)
x_line = np.linspace(X_test.min(), X_test.max(), 100).reshape(-1, 1)
plt.plot(x_line, model.predict(x_line), color='red', linewidth=2)
plt.xlabel('Source Bytes (sbytes)')
plt.ylabel('Connection Duration (seconds)')
plt.title('Linear Regression: Duration vs Source Bytes')
plt.show()

## Conclussion

This was only an introduction to show you the capabilities of a Jupyter Notebook. Throughout the course we will use some of these capabilities to prepare and protect our computer systems and networks.